In [1]:
import multiprocessing
import os

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(
    multiprocessing.cpu_count()
)

import batman
import blackjax
import gpjax as gpx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import numpy as np
import pandas as pd
from gpjax.kernels import RBF, Linear, Periodic
from jax import jit
from jaxoplanet import orbits
from jaxoplanet.light_curves import LimbDarkLightCurve
from jaxopt import ScipyMinimize
from tensorflow_probability.substrates.jax.bijectors import Log
from tensorflow_probability.substrates.jax.distributions import (
    Normal,
    TruncatedNormal,
    TransformedDistribution,
    Uniform,
)

from inference import log_likelihood_function
from kernels import OrnsteinUhlenbeck
from kernelsearch import KernelSearch, describe_kernel, get_trainables, set_trainables
from mcmc import nuts_warmup, run_mcmc
from util import calculate_example_lightcurve, lc_model

jax.config.update("jax_enable_x64", True)

rng_key = jax.random.PRNGKey(42)

Things to consider:
- heteroskedastic noise (supported natively -- check, or maybe not with fitting.. see if that important)
- doing kernel search per time series, or across all time series (loop over all datasets and calculate cumulative likelihood -> cummulative aic/bic)
- (normalise mean?)
- need longer time series data for proper learning

Questions:
- longer time series
- limb darkening fitting

## Load Data

In [2]:
df = pd.read_csv("data/lc2.dat", sep=" ", header=None)
df.columns = ["Time", "FWL", "e_FWL"] + [
    f"{prefix}FWB{i:02d}" for i in range(1, 27) for prefix in ("", "e_")
]
x = df["Time"].to_numpy()
x_min = np.amin(x)
x -= x_min

y = (
    df[
        [
            col
            for col in df.columns
            if "FWB" in col and "e_" not in col and "FWB20" not in col
        ]
    ]
    .to_numpy()
    .T
)
yerr = (
    df[[col for col in df.columns if "e_FWB" in col and "e_FWB20" not in col]]
    .to_numpy()
    .T
)

mask = np.ones_like(x, dtype=bool)
mask[7:41] = False

ld_pars = [
    (0.92, -0.0737),
    (0.87, -0.0523),
    (0.79, 0.0380),
    (0.73, 0.0726),
    (0.71, 0.0721),
    (0.67, 0.0837),
    (0.63, 0.1050),
    (0.59, 0.1241),
    (0.58, 0.1330),
    (0.58, 0.1209),
    (0.58, 0.1295),
    (0.55, 0.1336),
    (0.53, 0.1364),
    (0.50, 0.1512),
    (0.50, 0.1433),
    (0.48, 0.1446),
    (0.47, 0.1449),
    (0.45, 0.1452),
    (0.44, 0.1464),
    (0.42, 0.1477),
    (0.42, 0.1476),
    (0.40, 0.1482),
    (0.38, 0.1474),
    (0.37, 0.1488),
    (0.37, 0.1494),
]
u1_uncertainties = [0.02] * 6 + [0.01] * 19

## Get GP models for all light curves

In [3]:
kernel_library = [
    Linear(),
    RBF(),
    OrnsteinUhlenbeck(),
    Periodic(),
]

In [4]:
gps = {}
for i in range(len(y)):
    D = gpx.Dataset(x[mask].reshape(-1, 1), y[i][mask].reshape(-1, 1))

    tree = KernelSearch(
        kernel_library,
        X=jnp.array(x[mask]),
        y=jnp.array(y[i][mask]),
        obs_stddev=jnp.amax(yerr[i][mask]),
        verbosity=0,
    )

    model = tree.search(
        depth=7,
        n_leafs=2,
        patience=0,
    )
    print(describe_kernel(model))
    gps[i] = model

Periodic + Linear
OU
RBF
OU
RBF
OU
OU
OU
RBF
RBF
RBF
RBF
OU
RBF
RBF
RBF
RBF
RBF
OU
OU
Periodic
Periodic + (Periodic * Linear * Linear)
OU
RBF + OU
RBF


## Define LC model

In [5]:
def get_lc_model(ld_pars):
    def lc_model(t, params):
        central = orbits.keplerian.Central(
            mass=0.869,
            radius=0.8132,  # the radius in solar radii calculated as (0.05272 AU)/13.94, where 13.94 is the best fit a/R_star given in the paper
        )

        # The light curve calculation requires an orbit
        orbit = orbits.keplerian.Body(
            central=central,
            period=4.7423749,
            radius=params[0] * central.radius,
            inclination=jnp.deg2rad(87.6),
            time_transit=57983.20725000 - x_min,
        )

        lc = LimbDarkLightCurve([params[1], ld_pars[1]]).light_curve(orbit, t=t)
        return lc

    return lc_model

## Define likelihood, prior, posterior

In [6]:
def get_logprob(model, y, lc_model, ld_params, ld_params_unc):
    initial_position = {
        "gp_parameter": get_trainables(model, unconstrain=True),
        "lc_parameter": jnp.array([0.11250, ld_params[0]]),
    }

    param_priors = {
        "gp_parameter": Normal(
            loc=initial_position["gp_parameter"],
            scale=0.1 * jnp.abs(initial_position["gp_parameter"]),
        ),
        "lc_parameter": Normal(
            loc=initial_position["lc_parameter"],
            scale=[0.06, ld_params_unc],
        ),
    }

    log_likelihood = log_likelihood_function(
        model.unconstrain(),
        lc_model,
        x,
        y,
        mask,
        fix_gp=False,
        compile=True,
    )

    @jit
    def log_priors(params):
        gp_log_priors = param_priors["gp_parameter"].log_prob(params["gp_parameter"])
        lc_log_priors = param_priors["lc_parameter"].log_prob(params["lc_parameter"])
        return jnp.sum(gp_log_priors) + jnp.sum(lc_log_priors)

    @jit
    def log_probability(params):
        return log_likelihood(params) + log_priors(params)

    return log_probability, initial_position

## Fits

In [7]:
sols = []
for i in range(len(y)):
    lc_model = jit(get_lc_model(ld_pars[i]))
    log_probability, initial_position = get_logprob(
        gps[i],
        y[i],
        lc_model,
        ld_pars[i],
        u1_uncertainties[i],
    )

    lbfgsb = ScipyMinimize(
        fun=jit(lambda par: -log_probability(par)), method="l-bfgs-b"
    )
    lbfgsb_sol = lbfgsb.run(initial_position)
    sol = lbfgsb_sol.params
    print(sol["lc_parameter"])
    sols.append(sol)

[0.10834508 0.91766548]
[0.1075686  0.87447606]
[0.10974428 0.79344511]
[0.10946588 0.73913855]
[0.10782845 0.71834028]
[0.1118527 0.6809881]
[0.10896151 0.62998703]
[0.10972217 0.58955827]
[0.11258969 0.58035373]
[0.10869045 0.58083625]
[0.11295785 0.58180759]
[0.11066344 0.54898926]
[0.11103265 0.5314092 ]
[0.11072612 0.50064238]
[0.11213865 0.5016298 ]
[0.11188713 0.47935025]
[0.11246322 0.47297378]
[0.10940187 0.45103176]
[0.11414033 0.43987898]
[0.11066378 0.42030608]
[0.11242822 0.42095546]
[0.10990961 0.39926098]
[0.10033251 0.38034596]
[0.10680085 0.37036465]
[0.11717684 0.37001005]


## Do MCMC

In [ ]:
# Adapted from BlackJax's introduction notebook.
num_adapt = 400
num_samples = 400
num_chains = 8

chains = []

for i in range(len(y)):
    lc_model = jit(get_lc_model(ld_pars[i]))
    log_probability, initial_position = get_logprob(
        gps[i], y[i], lc_model, ld_pars[i], u1_uncertainties[i]
    )

    rng_key, warmup_key, sample_key = jax.random.split(rng_key, 3)

    state, parameters = nuts_warmup(
        warmup_key,
        log_probability,
        initial_position,
        num_steps=num_adapt,
    )

    initial_positions = {
        "gp_parameter": jnp.tile(state.position["gp_parameter"], (num_chains, 1)),
        "lc_parameter": jnp.tile(state.position["lc_parameter"], (num_chains, 1)),
    }

    final_state, state_history, info_history = run_mcmc(
        sample_key,
        log_probability,
        parameters,
        initial_positions,
        num_steps=num_samples,
    )

    chain = np.array(
        state_history.position["lc_parameter"].reshape(
            -1, state_history.position["lc_parameter"].shape[-1]
        )
    )
    chains.append(chain)

Running window adaptation



Running window adaptation



Running window adaptation



Running window adaptation



Running window adaptation


...


Running window adaptation



Running window adaptation



Running window adaptation



Running window adaptation



Running window adaptation
